In [ ]:
import time
import pandas as pd
from serpapi import GoogleSearch
from langdetect import detect
import re
from google.colab import drive

In [ ]:
SERPAPI_KEY = "206de452779a89f309c6a1d6b24b009ff9daca88320e09389fa2e7ae3007400b"
HOTEL_QUERIES = [
    "ไมด้า แกรนด์ ทวารวดี นครปฐม",
    "โรงแรมไมด้า แกรนด์ ทวารวดี นครปฐม",
    "Mida Grande Hotel Dhavaravati Nakhon Pathom",
]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def find_hotel_data_id(api_key):
    for q in HOTEL_QUERIES:
        print(f"\n🔍 ลองค้นหา: {q}")
        params = {
            "engine": "google_maps",
            "type": "search",
            "q": q,
            "hl": "th",
            "api_key": api_key,
        }
        data = GoogleSearch(params).get_dict()


        local = data.get("local_results")
        if isinstance(local, list) and local:
            print("เจอ local_results", len(local), "รายการ")
            place = local[0]
            print(" เลือก:", place.get("title"))
            print("data_id:", place.get("data_id"))
            return place.get("data_id"), place.get("title")

        # สำรอง: place_results
        place = data.get("place_results")
        if isinstance(place, dict):
            print("เจอ place_results (dict)")
            print("เลือก:", place.get("title"))
            return place.get("data_id"), place.get("title")
        elif isinstance(place, list) and place:
            print("เจอ place_results (list)")
            p0 = place[0]
            print(" เลือก:", p0.get("title"))
            return p0.get("data_id"), p0.get("title")

    raise RuntimeError("หา data_id โรงแรมไมด้า แกรนด์ ทวารวดี ไม่เจอ")


In [ ]:
def extract_review_text(r):
    # เผื่อบางครั้ง text อยู่ field อื่น
    candidates = [
        r.get("text"),
        r.get("snippet"),
        r.get("description"),
        r.get("summary"),
        r.get("comment"),
    ]
    for c in candidates:
        if c and isinstance(c, str) and c.strip():
            return c.strip()
    return ""

def extract_review_language(r, text_value):
    lang = r.get("language")
    if lang and isinstance(lang, str) and lang.strip():
        return lang.strip()
    if text_value:
        try:
            return detect(text_value)
        except:
            return "unknown"
    return "unknown"

def extract_review_time(r):
    candidates = [
        r.get("relative_time_description"),
        r.get("date"),
    ]
    for c in candidates:
        if c and isinstance(c, str) and c.strip():
            return c.strip()
    return ""

In [ ]:
def fetch_reviews_with_text(api_key, max_reviews=1000):
    data_id, hotel_name = find_hotel_data_id(api_key)
    print(f"\n ใช้ data_id: {data_id}")
    print(f" โรงแรม: {hotel_name}")

    all_reviews = []
    next_page_token = None
    page = 1

    while len(all_reviews) < max_reviews:
        print(f"\nดึงหน้า {page} ... (ตอนนี้มี {len(all_reviews)} รีวิวที่มีข้อความ)")

        params = {
            "engine": "google_maps_reviews",
            "api_key": api_key,
            "data_id": data_id,
            "hl": "th",
            "sort_by": "newest",
        }
        if next_page_token:
            params["next_page_token"] = next_page_token

        result = GoogleSearch(params).get_dict()
        reviews = result.get("reviews", [])

        if not reviews:
            print("ไม่มีรีวิวเพิ่มแล้ว")
            break

        for r in reviews:
            text_value = extract_review_text(r)

            # เอาเฉพาะรีวิวที่มีข้อความจริง
            if text_value and text_value.strip() != "":
                all_reviews.append({
                    "ชื่อโรงแรม": hotel_name,
                    "ชื่อผู้รีวิว": (r.get("user") or r.get("author") or "").get("name")
                                      if isinstance(r.get("user"), dict) else r.get("user") or r.get("author"),
                    "เรตติ้ง": r.get("rating"),
                    "ข้อความรีวิว": text_value,
                    "ภาษารีวิว": extract_review_language(r, text_value),
                    "เวลารีวิว": extract_review_time(r),
                })

            if len(all_reviews) >= max_reviews:
                break

        next_page_token = result.get("serpapi_pagination", {}).get("next_page_token")
        if not next_page_token:
            print("ไม่มีหน้าถัดไปแล้ว")
            break

        page += 1
        time.sleep(2)   # กัน rate limit

    print(f"\nได้รีวิวที่มีข้อความทั้งหมด {len(all_reviews)} รายการ")
    return pd.DataFrame(all_reviews)


In [ ]:
bronze_df = fetch_reviews_with_text(SERPAPI_KEY, max_reviews=1000)

save_path = "/content/drive/MyDrive/DST/Bronze.csv"
bronze_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"\n บันทึกไฟล์ Bronze เรียบร้อยที่: {save_path}")
bronze_df.head()


🔍 ลองค้นหา: ไมด้า แกรนด์ ทวารวดี นครปฐม
เจอ place_results (dict)
เลือก: โรงแรมไมด้า แกรนด์ ทวารวดี

 ใช้ data_id: 0x30e2e5dbd116d2fb:0xed0fb80b637589f8
 โรงแรม: โรงแรมไมด้า แกรนด์ ทวารวดี

ดึงหน้า 1 ... (ตอนนี้มี 0 รีวิวที่มีข้อความ)

ดึงหน้า 2 ... (ตอนนี้มี 8 รีวิวที่มีข้อความ)

ดึงหน้า 3 ... (ตอนนี้มี 18 รีวิวที่มีข้อความ)

ดึงหน้า 4 ... (ตอนนี้มี 28 รีวิวที่มีข้อความ)

ดึงหน้า 5 ... (ตอนนี้มี 38 รีวิวที่มีข้อความ)

ดึงหน้า 6 ... (ตอนนี้มี 48 รีวิวที่มีข้อความ)

ดึงหน้า 7 ... (ตอนนี้มี 58 รีวิวที่มีข้อความ)

ดึงหน้า 8 ... (ตอนนี้มี 68 รีวิวที่มีข้อความ)

ดึงหน้า 9 ... (ตอนนี้มี 78 รีวิวที่มีข้อความ)

ดึงหน้า 10 ... (ตอนนี้มี 88 รีวิวที่มีข้อความ)

ดึงหน้า 11 ... (ตอนนี้มี 98 รีวิวที่มีข้อความ)

ดึงหน้า 12 ... (ตอนนี้มี 108 รีวิวที่มีข้อความ)

ดึงหน้า 13 ... (ตอนนี้มี 118 รีวิวที่มีข้อความ)

ดึงหน้า 14 ... (ตอนนี้มี 128 รีวิวที่มีข้อความ)

ดึงหน้า 15 ... (ตอนนี้มี 138 รีวิวที่มีข้อความ)

ดึงหน้า 16 ... (ตอนนี้มี 148 รีวิวที่มีข้อความ)

ดึงหน้า 17 ... (ตอนนี้มี 158 รีวิวที่มีข้อความ)


,ชื่อโรงแรม,ชื่อผู้รีวิว,เรตติ้ง,ข้อความรีวิว,ภาษารีวิว,เวลารีวิว
0,โรงแรมไมด้า แกรนด์ ทวารวดี,Lakfantiew Youandme,5.0,Mida Grande Hotel Dhavaravati Nakhon Pathom โร...,th,เดือนที่แล้ว
1,โรงแรมไมด้า แกรนด์ ทวารวดี,Suntharee Udomphon,4.0,ห้องพักโดยรวมโอเครค่ะ ห้องน้ำกว้างมากสะดวกสบาย...,th,สัปดาห์ที่แล้ว
2,โรงแรมไมด้า แกรนด์ ทวารวดี,สิงห์สนุก ซ่าส์,5.0,เป็นโรงแรมที่สวยสะอาดราคาไม่แพงแต่คุณภาพแรงคับ...,th,เดือนที่แล้ว
3,โรงแรมไมด้า แกรนด์ ทวารวดี,Gam Supatta,1.0,โรงแรมสวยบรรยากาศดี แต่ได้เข้าไปใช้บริการนวดมา...,th,เดือนที่แล้ว
4,โรงแรมไมด้า แกรนด์ ทวารวดี,Thitipa Chanapai,4.0,เมื่อวานไปชมวิวสกายบาร์ที่โรงแรมนี้ถือว่าดีงาม...,th,เดือนที่แล้ว
